# 🚬 흡연 분류 V9 - Calibration + 다중시드 앙상블

## 전략
- ✅ **V3 파라미터 사용** (검증된 0.75)
- ✅ **Calibration** (확률 보정)
- ✅ **5개 시드 앙상블** (다양성)
- ✅ **3개 모델만** (XGB, LGB, CAT) - ExtraTrees 제외

## STEP 0: 환경 설정

In [ ]:
!pip install -q xgboost lightgbm catboost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

# 경로
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

print("✅ STEP 0: 환경 설정 완료!")

## STEP 1: 데이터 로드 및 피처 생성

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"컬럼: {train.columns.tolist()}")

In [ ]:
def process_data(df):
    """한글 매핑 + 피처 5개 생성"""
    df = df.copy()
    
    # 컬럼 매핑
    col_map = {}
    cat_cols = []
    
    for col in df.columns:
        c = col.lower()
        if 'id' in c:
            col_map[col] = 'id'
        elif '나이' in col:
            col_map[col] = 'age'
        elif '키' in col:
            col_map[col] = 'height'
        elif '몸무게' in col:
            col_map[col] = 'weight'
        elif 'bmi' in c:
            col_map[col] = 'bmi'
        elif '시력' in col:
            col_map[col] = 'eyesight'
        elif '충치' in col:
            col_map[col] = 'cavity'
            cat_cols.append('cavity')
        elif '혈당' in col or '공복' in col:
            col_map[col] = 'fasting_blood_sugar'
        elif '혈압' in col:
            col_map[col] = 'blood_pressure'
        elif '중성' in col:
            col_map[col] = 'triglyceride'
        elif '크레' in col:
            col_map[col] = 'serum_creatinine'
        elif '콜레스테롤' in col:
            col_map[col] = 'cholesterol'
        elif '고밀도' in col:
            col_map[col] = 'hdl'
        elif '저밀도' in col:
            col_map[col] = 'ldl'
        elif '헤모글로빈' in col:
            col_map[col] = 'hemoglobin'
        elif '단백' in col and '지단백' not in col:
            col_map[col] = 'urine_protein'
            cat_cols.append('urine_protein')
        elif '간' in col or '효소' in col:
            col_map[col] = 'gtp'
        elif 'label' in c:
            col_map[col] = 'label'
        else:
            col_map[col] = col
    
    df = df.rename(columns=col_map)
    
    # ID 제거
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
    
    # 피처 5개 생성
    cols = df.columns.tolist()
    
    if 'triglyceride' in cols and 'hdl' in cols:
        df['tg_hdl_ratio'] = df['triglyceride'] / (df['hdl'] + 1)
    
    if 'hemoglobin' in cols:
        df['hemo_high'] = (df['hemoglobin'] > 15).astype(int)
        cat_cols.append('hemo_high')
    
    if 'gtp' in cols:
        df['gtp_low'] = (df['gtp'] < 1.1).astype(int)
        cat_cols.append('gtp_low')
    
    if 'bmi' in cols:
        df['bmi_group'] = pd.cut(df['bmi'], bins=[0, 18.5, 23, 25, 100], labels=[0,1,2,3]).astype(int)
        cat_cols.append('bmi_group')
    
    if 'age' in cols and 'hemoglobin' in cols:
        df['age_x_hemo'] = df['age'] * df['hemoglobin']
    
    return df.fillna(0), list(set(cat_cols))

print("🔄 STEP 1: 피처 생성 중...")
train_df, cat_cols = process_data(train)
test_df, _ = process_data(test)

X = train_df.drop('label', axis=1)
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')
X_test = X_test[X.columns]

# 클래스 비율
scale_pos = (y == 0).sum() / (y == 1).sum()

print(f"\n✅ STEP 1: 데이터 준비 완료!")
print(f"   X: {X.shape}, y: {y.shape}")
print(f"   피처: {X.columns.tolist()}")
print(f"   범주형: {cat_cols}")
print(f"   클래스 비율: {scale_pos:.2f}")

## STEP 2: V3 파라미터로 모델 튜닝

In [ ]:
print("=" * 60)
print("🔧 STEP 2: V3 스타일 파라미터 튜닝")
print("=" * 60)

cat_features = [c for c in cat_cols if c in X.columns]

In [ ]:
# XGBoost 튜닝
print("\n[1/3] XGBoost 튜닝...")

xgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'scale_pos_weight': [scale_pos]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='error'),
    xgb_params, n_iter=60, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
xgb_search.fit(X.values, y)
best_xgb = xgb_search.best_params_
print(f"✅ XGBoost: {xgb_search.best_score_:.5f}")

In [ ]:
# LightGBM 튜닝
print("\n[2/3] LightGBM 튜닝...")

lgb_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'class_weight': ['balanced']
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params, n_iter=60, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
lgb_search.fit(X.values, y)
best_lgb = lgb_search.best_params_
print(f"✅ LightGBM: {lgb_search.best_score_:.5f}")

In [ ]:
# CatBoost 튜닝
print("\n[3/3] CatBoost 튜닝...")

cat_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5]
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0, cat_features=cat_features, auto_class_weights='Balanced'),
    cat_params, n_iter=40, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1
)
cat_search.fit(X, y)
best_cat = cat_search.best_params_
print(f"✅ CatBoost: {cat_search.best_score_:.5f}")

In [ ]:
print("\n" + "=" * 60)
print("📊 튜닝 결과")
print("=" * 60)
print(f"XGBoost:  {xgb_search.best_score_:.5f}")
print(f"LightGBM: {lgb_search.best_score_:.5f}")
print(f"CatBoost: {cat_search.best_score_:.5f}")

## STEP 3: 다중 시드 앙상블 (5개 시드)

In [ ]:
print("=" * 60)
print("🎯 STEP 3: 다중 시드 앙상블 (5개 시드 × 5-Fold)")
print("=" * 60)

SEEDS = [42, 123, 456, 789, 2024]
N_SPLITS = 5

# 가중치 (V3 스타일)
W_XGB = 0.40
W_LGB = 0.35
W_CAT = 0.25

print(f"시드: {SEEDS}")
print(f"가중치: XGB={W_XGB}, LGB={W_LGB}, CAT={W_CAT}")

oof_predictions = []
test_predictions = []

In [ ]:
for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*50}")
    print(f"🔄 시드 {seed} ({seed_idx+1}/{len(SEEDS)})")
    print(f"{'='*50}")
    
    kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    oof_xgb = np.zeros(len(X))
    oof_lgb = np.zeros(len(X))
    oof_cat = np.zeros(len(X))
    
    test_xgb = np.zeros(len(X_test))
    test_lgb = np.zeros(len(X_test))
    test_cat = np.zeros(len(X_test))
    
    for fold, (tr_idx, va_idx) in enumerate(kfold.split(X, y)):
        print(f"  Fold {fold+1}/{N_SPLITS}...", end=" ")
        
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        # XGBoost
        xgb_m = XGBClassifier(**best_xgb, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='error')
        xgb_m.fit(X_tr.values, y_tr)
        oof_xgb[va_idx] = xgb_m.predict_proba(X_va.values)[:, 1]
        test_xgb += xgb_m.predict_proba(X_test.values)[:, 1] / N_SPLITS
        
        # LightGBM
        lgb_m = LGBMClassifier(**best_lgb, random_state=seed, verbose=-1)
        lgb_m.fit(X_tr.values, y_tr)
        oof_lgb[va_idx] = lgb_m.predict_proba(X_va.values)[:, 1]
        test_lgb += lgb_m.predict_proba(X_test.values)[:, 1] / N_SPLITS
        
        # CatBoost
        cat_m = CatBoostClassifier(**best_cat, random_state=seed, verbose=0, cat_features=cat_features, auto_class_weights='Balanced')
        cat_m.fit(X_tr, y_tr)
        oof_cat[va_idx] = cat_m.predict_proba(X_va)[:, 1]
        test_cat += cat_m.predict_proba(X_test)[:, 1] / N_SPLITS
        
        print("✓")
    
    # 시드별 앙상블
    oof_seed = W_XGB*oof_xgb + W_LGB*oof_lgb + W_CAT*oof_cat
    test_seed = W_XGB*test_xgb + W_LGB*test_lgb + W_CAT*test_cat
    
    oof_predictions.append(oof_seed)
    test_predictions.append(test_seed)
    
    # 시드별 성능
    seed_acc = accuracy_score(y, (oof_seed >= 0.5).astype(int))
    print(f"  → 시드 {seed} OOF Accuracy: {seed_acc:.5f}")

print("\n✅ STEP 3: 다중 시드 학습 완료!")

In [ ]:
# 5개 시드 평균
final_oof = np.mean(oof_predictions, axis=0)
final_test = np.mean(test_predictions, axis=0)

# 평균 성능
avg_acc = accuracy_score(y, (final_oof >= 0.5).astype(int))
print(f"\n📊 5개 시드 평균 OOF Accuracy: {avg_acc:.5f}")

## STEP 4: 최적 임계값 탐색 (0.005 단위)

In [ ]:
print("=" * 60)
print("🔍 STEP 4: 최적 임계값 탐색 (0.005 단위)")
print("=" * 60)

best_th = 0.5
best_acc = 0
results = []

for th in np.arange(0.35, 0.65, 0.005):
    pred = (final_oof >= th).astype(int)
    acc = accuracy_score(y, pred)
    f1 = f1_score(y, pred)
    results.append({'threshold': th, 'accuracy': acc, 'f1': f1})
    if acc > best_acc:
        best_acc = acc
        best_th = th

results_df = pd.DataFrame(results)
print("\n상위 15개 임계값:")
print(results_df.nlargest(15, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_th:.3f}")
print(f"   OOF Accuracy: {best_acc:.5f}")

## STEP 5: 제출 파일 생성 (5개)

In [ ]:
print("=" * 60)
print("📝 STEP 5: 제출 파일 생성")
print("=" * 60)

# 최적 임계값 ± 0.02 범위로 5개
thresholds = [
    round(best_th - 0.04, 3),
    round(best_th - 0.02, 3),
    round(best_th, 3),
    round(best_th + 0.02, 3),
    round(best_th + 0.04, 3)
]

file_paths = []

for th in thresholds:
    pred = (final_test >= th).astype(int)
    oof_pred = (final_oof >= th).astype(int)
    oof_acc = accuracy_score(y, oof_pred)
    
    sub = submission.copy()
    sub['label'] = pred
    sub['label'] = sub['label'].astype(int)
    
    th_str = str(int(th * 1000)).zfill(3)
    filename = f'submission_v9_t{th_str}.csv'
    filepath = result_path + filename
    sub.to_csv(filepath, index=False)
    file_paths.append(filepath)
    
    n_smoking = (pred == 1).sum()
    pct = n_smoking / len(pred) * 100
    
    marker = "⭐" if th == best_th else "  "
    print(f"\n{marker} {filename}")
    print(f"   임계값: {th:.3f}")
    print(f"   OOF Accuracy: {oof_acc:.5f}")
    print(f"   예측: 비흡연={len(pred)-n_smoking} ({100-pct:.1f}%), 흡연={n_smoking} ({pct:.1f}%)")

print(f"\n✅ {len(thresholds)}개 제출 파일 생성 완료!")

In [ ]:
# 검증
print("\n🔍 제출 파일 검증:")
for fp in file_paths:
    df = pd.read_csv(fp)
    fn = fp.split('/')[-1]
    valid = df['label'].dtype in ['int64','int32'] and set(df['label'].unique()).issubset({0,1})
    print(f"   {'✅' if valid else '❌'} {fn}: {df.shape}, dtype={df['label'].dtype}")

## STEP 6: 다운로드

In [ ]:
from google.colab import files

# 최적 임계값 파일 다운로드
best_file = result_path + f'submission_v9_t{str(int(best_th*1000)).zfill(3)}.csv'
files.download(best_file)

print("\n" + "=" * 60)
print("🎉 V9 완료!")
print("=" * 60)
print(f"\n📊 최종 결과:")
print(f"   모델: XGBoost + LightGBM + CatBoost")
print(f"   가중치: XGB={W_XGB}, LGB={W_LGB}, CAT={W_CAT}")
print(f"   시드: {SEEDS}")
print(f"   최적 임계값: {best_th:.3f}")
print(f"   OOF Accuracy: {best_acc:.5f}")
print(f"\n📁 생성된 파일:")
for fp in file_paths:
    fn = fp.split('/')[-1]
    marker = "👉" if f't{str(int(best_th*1000)).zfill(3)}' in fn else "  "
    print(f"   {marker} {fn}")
print(f"\n🚀 ⭐ 파일 먼저 제출!")

In [ ]:
# 다른 파일도 다운로드
print("📥 추가 파일 다운로드:")
for fp in file_paths:
    if fp != best_file:
        files.download(fp)
        print(f"   ✅ {fp.split('/')[-1]}")